# Assignment 1

## Extract webpage data given the url

Create a Python script that performs basic web scraping on a page to extract all the information into text and returns it as a string. String should not contain tags.

In [17]:
import re
import requests
from bs4 import BeautifulSoup

def parse_web_page(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    clean_text = re.sub(r"\s+", " ", soup.get_text())

    return clean_text

print(parse_web_page('https://github.com/jadatix')[:255])
print(parse_web_page('https://en.wikipedia.org/wiki/Web_scraping')[:255])

 jadatix (Andrii) · GitHub Skip to content Navigation MenuSign inAppearance settingsPlatformAI CODE CREATIONGitHub CopilotWrite better code with AIGitHub Copilot appDirect agents from issue to mergeMCP RegistryIntegrate external toolsDEVELOPER WORKFLOWSAc
 Web scraping - Wikipedia Jump to content Main menu Main menu move to sidebar hide Navigation Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us Contribute HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages Search 


## Get data from the API

Create a python script that performs basic request to API endpoint and saves that data to a JSON file result.json.

In [18]:
import json
import requests

def parse_api(api_url):
    response = requests.get(api_url)
    response.raise_for_status()

    data = response.json()
    with open("output/result.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

parse_api('https://catfact.ninja/breeds')

## Parse the json file

Parse the `weather.json` file and return weather data for a specific date, that is given as a parameter. Return the data as an array.

In [19]:
import json

def parse_json(date):
    with open("resources/weather.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    daily_records = data.get("daily", [])
    result = [day for day in daily_records if day.get("date") == date]

    return result
    
target_date = '2024-08-19'
print(parse_json(target_date))

[{'date': '2024-08-19', 'max_temperature': 30.0, 'min_temperature': 21.0, 'precipitation': 5.0, 'wind_speed': 10.0, 'humidity': 70, 'weather_description': 'Light rain'}]


## Parse the csv file

Parse the `weather.csv` file and return weather data for a specific date, that is given as a parameter. Return the data as an array.

In [20]:
import csv

def parse_csv(date):
    result = []

    with open("resources/weather.csv", mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("CET") == date or row.get("Date") == date:
                result.append(row)

    return result
    
target_date = '1997-5-22'
print(parse_csv(target_date))

[{'CET': '1997-5-22', 'Max TemperatureC': '25', 'Mean TemperatureC': '18', 'Min TemperatureC': '10', 'Dew PointC': '11', 'MeanDew PointC': '8', 'Min DewpointC': '6', 'Max Humidity': '88', ' Mean Humidity': '54', ' Min Humidity': '34', ' Max Sea Level PressurehPa': '1017', ' Mean Sea Level PressurehPa': '1015', ' Min Sea Level PressurehPa': '1012', ' Max VisibilityKm': '10', ' Mean VisibilityKm': '10', ' Min VisibilitykM': '10', ' Max Wind SpeedKm/h': '11', ' Mean Wind SpeedKm/h': '3', ' Max Gust SpeedKm/h': '', 'Precipitationmm': '0.00', ' CloudCover': '3', ' Events': '', 'WindDirDegrees': '277'}]


## Visualize data

Visualize the weather.csv data using matplotlib. Choose your own approach to data visualization. Save the results (as .png, .webp files etc., your choise) in this repository.

In [21]:
import matplotlib.pyplot as plt
import pandas as pd


def visualize_data():
    df = pd.read_csv("resources/weather.csv")

    df["CET"] = pd.to_datetime(df["CET"])
    df = df.sort_values("CET")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        "Weather Data Visualization",
        fontsize=14,
        fontweight="bold",
    )

    axes[0].plot(
        df["CET"],
        df["Mean TemperatureC"],
        color="tab:blue",
        linewidth=1.5,
        label="Mean Temp (°C)",
    )
    axes[0].set_title("Line Plot: Sequential Mean Temperature Over Time")
    axes[0].set_xlabel("Date (CET)")
    axes[0].set_ylabel("Temperature (°C)")
    axes[0].grid(True, linestyle="--", alpha=0.5)

    axes[1].hist(
        df["Mean TemperatureC"].dropna(),
        bins=30,
        color="tab:orange",
        edgecolor="black",
        alpha=0.7,
    )
    axes[1].set_title("Histogram: Distribution of Mean Temperature")
    axes[1].set_xlabel("Mean Temperature (°C)")
    axes[1].set_ylabel("Count")
    axes[1].grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()

    plt.savefig("output/weather_visualization.png", dpi=300)
    plt.close()

    return None

visualize_data()

## Data imputation and normalization.

Download kaggle dataset with incomplete/missing data, and try data imputation techniques (mean, median, mode, knn) on that data. Normalize the data with best option for each dataset (minmax, z-score, robust). Visualize the data (choose your own approach to data visualization).

Dataset options:
- Different Types of Missing Data (by Suryaprabha19) https://www.kaggle.com/datasets/suryaprabha19/different-types-of-missing-data
- Retail Product Dataset with Missing Values (by Himel Sarder) https://www.kaggle.com/datasets/himelsarder/retail-product-dataset-with-missing-values
- Data Cleaning - Feature Imputation (by 
Mr.Machine) https://www.kaggle.com/datasets/ilayaraja07/data-cleaning-feature-imputation
- BI intro to data cleaning eda and machine learning (by Walekhwa Tambiti Leo Philip) https://www.kaggle.com/datasets/walekhwatlphilip/intro-to-data-cleaning-eda-and-machine-learning

In [22]:
# pip install "kagglehub[pandas-datasets]"

import kagglehub
from kagglehub import KaggleDatasetAdapter
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

def load_dataset(url, csv_file):
    df = kagglehub.dataset_load(
            KaggleDatasetAdapter.PANDAS,
            url,
            csv_file
        )
    return df

def impute_data(df):
    df_imputed = df.copy()

    numerical_cols = df_imputed.select_dtypes(include=["number"]).columns
    categorical_cols = df_imputed.select_dtypes(exclude=["number"]).columns

    for col in numerical_cols:
        if df_imputed[col].isnull().any():
            df_imputed[col].fillna(df_imputed[col].mean(), inplace=True)

    for col in categorical_cols:
        if df_imputed[col].isnull().any():
            df_imputed[col].fillna(df_imputed[col].mode()[0], inplace=True)

    return df_imputed


def normalize_data(df):
    numerical_cols = df.select_dtypes(include=["number"])
    categorical_cols = df.select_dtypes(exclude=["number"])

    scaler = MinMaxScaler()

    numerical_scaled = scaler.fit_transform(numerical_cols)

    numerical_scaled_df = pd.DataFrame(
        numerical_scaled, columns=numerical_cols.columns
    )

    if not categorical_cols.empty:
        df_scaled = pd.concat(
            [numerical_scaled_df, categorical_cols.reset_index(drop=True)],
            axis=1,
        )
    else:
        df_scaled = numerical_scaled_df

    return df_scaled


def visualize_data(df):
    num_cols = df.select_dtypes(include=[np.number]).columns

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(
        "Data Visualization", fontsize=14
    )

    for col in num_cols:
        axes[0].hist(df[col].dropna(), bins=30, alpha=0.5, label=col)

    axes[0].set_title("Histograms (Numerical Distribution)")
    axes[0].set_xlabel("Value")
    axes[0].set_ylabel("Count")
    axes[0].legend(loc="upper right")
    axes[0].grid(True, linestyle="--", alpha=0.5)

    if len(num_cols) >= 2:
        col_x, col_y = num_cols[0], num_cols[1]
        axes[1].scatter(df[col_x], df[col_y], alpha=0.6, edgecolors="none")
        axes[1].set_title(f"Scatter Plot ({col_x} vs {col_y})")
        axes[1].set_xlabel(col_x)
        axes[1].set_ylabel(col_y)
        axes[1].grid(True, linestyle="--", alpha=0.5)
    else:
        axes[1].plot(df[num_cols[0]].values)
        axes[1].set_title("Line Plot (Sequential Data)")
        axes[1].set_xlabel("Index")
        axes[1].set_ylabel(num_cols[0])

    plt.tight_layout()
    plt.savefig("output/imputation_normalization_results.png", dpi=300)
    plt.close()

    return None

df = load_dataset('ilayaraja07/data-cleaning-feature-imputation', 'Students_Performance_knn.csv')
df = impute_data(df)
df = normalize_data(df)
visualize_data(df)

/var/folders/xr/vbh1snqd5nv8kcvb9b32c7lc0000gn/T/ipykernel_13891/1311964754.py:30: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_imputed[col].fillna(df_imputed[col].mode()[0], inplace=True)
